# MyDigitalTwin — K-Means Clustering

**Objectif** : Construire des clusters de centres d'intérêts data-driven pour remplacer le keyword matching de la home page.

**Deux axes** :
- **Partie A — Content Clustering** : TF-IDF sur tous les textes (titres YouTube, recherches Google, artistes Spotify, films Netflix, produits Amazon...) → K-Means → top termes par cluster
- **Partie B — Behavioral Clustering** : heure, jour, plateforme → K-Means → profils comportementaux
- **Partie C — Fusion** : combiner les deux pour créer `interest_profiles` (lu par la home page)

**Outputs Delta** :
- `warehouse/content_clusters`
- `warehouse/behavioral_clusters`
- `warehouse/interest_profiles`

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, ArrayType

spark = SparkSession.builder \
    .appName("MyDigitalTwin-Clustering") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Paths Docker vs local
if os.path.exists("/opt/spark/warehouse"):
    WAREHOUSE = "/opt/spark/warehouse"
else:
    # Windows local (chemin absolu)
    WAREHOUSE = os.path.abspath(os.path.join(os.getcwd(), "../../../scripts/notebooks", "..", "..", "warehouse"))
    if not os.path.exists(WAREHOUSE):
        WAREHOUSE = "C:/Users/arnau/Documents/MyDigitalTwin/warehouse"

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

Warehouse: /opt/spark/warehouse


26/04/05 01:43:22 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE A — Content Clustering

On rassemble tout le texte consommé (titres, requêtes, artistes, produits) en une seule colonne `text`, puis on applique TF-IDF + KMeans.

In [2]:
# ── A1. CHARGEMENT DES SOURCES TEXTE ──────────────────────────────────────────

def read_table(table_name):
    """Lit une table Delta (parquet files) depuis le warehouse."""
    path = os.path.join(WAREHOUSE, table_name)
    return spark.read.parquet(path)

# YouTube — titres de vidéos regardées
yt_watch = read_table("youtube_watch")     .select(F.col("title").alias("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("youtube").alias("platform"),
            F.col("interaction_weight").alias("weight"))

# YouTube Searches — requêtes
yt_search = read_table("youtube_searches")     .select(F.col("title").alias("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("youtube").alias("platform"),
            F.lit(1.0).cast(FloatType()).alias("weight"))

# Google Searches — requêtes
g_search = read_table("google_searches")     .select(F.col("query").alias("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("google").alias("platform"),
            F.lit(1.0).cast(FloatType()).alias("weight"))

# Spotify — artiste + titre
spotify = read_table("spotify_streams") \
      .select(F.col("artistName").alias("text"),
              F.col("listen_hour").alias("hour"),
              F.col("listen_weekday").alias("weekday"),
              F.lit("spotify").alias("platform"),
              F.col("interaction_weight").alias("weight")) \
      .dropDuplicates(["text"])

# Netflix — titre de l'émission
netflix = read_table("netflix_views")     .select(F.col("show_title").alias("text"),
            F.lit(21).cast(IntegerType()).alias("hour"),
            F.col("watch_weekday").alias("weekday"), F.lit("netflix").alias("platform"),
            F.col("interaction_weight").alias("weight"))

# Amazon — nom produit + catégorie
amazon = read_table("amazon_orders") \
      .select(F.col("category").alias("text"),   # ← plus de product_name
              F.col("order_hour").alias("hour"),
              F.col("order_weekday").alias("weekday"),
              F.lit("amazon").alias("platform"),
              F.col("interaction_weight").alias("weight")) \
      .dropDuplicates(["text"])

# Apple App Installs — pas de interaction_weight dans cette table
apple = read_table("apple_app_installs")     .withColumn("text", F.concat_ws(" ", F.col("app_name"), F.col("category")))     .select(F.col("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("apple").alias("platform"),
            F.lit(1.0).cast(FloatType()).alias("weight"))

# Google Chrome — titres de pages visitées
chrome = read_table("google_chrome")     .select(F.col("title").alias("text"), F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"), F.lit("chrome").alias("platform"),
            F.lit(1.5).cast(FloatType()).alias("weight"))

# ── Union de toutes les sources
all_text = yt_watch.union(yt_search).union(g_search).union(spotify)     .union(netflix).union(amazon).union(apple).union(chrome)

# Nettoyage : supprimer les nulls et les textes trop courts
all_text = all_text.filter(
    F.col("text").isNotNull() &
    (F.length(F.col("text")) > 3)
)

print(f"Total events : {all_text.count():,}")
all_text.show(5, truncate=50)


Total events : 82,307
+--------------------------------------------------+----+-------+--------+------+
|                                              text|hour|weekday|platform|weight|
+--------------------------------------------------+----+-------+--------+------+
|       https://www.youtube.com/watch?v=RrrlT6MnKPY|  19|      7| youtube|   1.0|
|INH INH Security 315969 TrustNewBetTotalBrandBr...|  19|      7| youtube|   1.0|
|          Pourquoi tu ne trouveras jamais l’amour.|  19|      7| youtube|   1.0|
|                BNPPF_HY4 Boost_9x16_FR_12sec_2505|  19|      7| youtube|   1.0|
|    Miss Gauff cette reine ! #tennis #rolandgarros|  19|      7| youtube|   1.0|
+--------------------------------------------------+----+-------+--------+------+
only showing top 5 rows



In [10]:
# ── A1b. NETTOYAGE ANTI-POLLUTION ─────────────────────────────────────────────
import re
from pyspark.sql.window import Window

# 1. Mots néerlandais courants (pages AliExpress BE, Chrome néerlandais)
NL_STOPWORDS = [
    "van", "voor", "met", "een", "het", "de", "en", "naar", "bij", "op",
    "uit", "als", "ook", "maar", "zijn", "wordt", "wordt", "meer", "alle",
    "winkelen", "populaire", "speelgoed"
]

# 2. Regex : patterns d'ads YouTube (codes internes INH/ING/Belfius)
AD_PATTERN = re.compile(
    r'(INH|VID\s+16x9|16x9\s+\d|TrustNew|BrandBranch|DailyBanking|'
    r'UnpackingMetal|ContactlessComp|FreeBankAccount)',
    re.IGNORECASE
)

def is_clean(text):
    if not text:
        return False
    t = text.lower()
    # Filtre 1 : ads YouTube (patterns internes)
    if AD_PATTERN.search(text):
        return False
    # Filtre 2 : texte majoritairement néerlandais (>2 mots NL sur les 6 premiers)
    words = t.split()[:8]
    if sum(1 for w in words if w in NL_STOPWORDS) >= 2:
        return False
    # Filtre 3 : texte trop long = description produit Amazon (> 120 chars)
    if len(text) > 120:
        return False
    if "gmail" in t or "mail.google" in t or "@" in t:
        return False
    return True

is_clean_udf = F.udf(is_clean, "boolean")

before = all_text.count()
all_text = all_text.filter(is_clean_udf(F.col("text")))

# Filtre 4 : dédupliquer les textes identiques (ads répétées à l'identique)
all_text = all_text.dropDuplicates(["text"])

w = Window.partitionBy("platform").orderBy(F.rand(seed=42))
all_text = all_text.withColumn("rn", F.row_number().over(w)) \
                     .filter(F.col("rn") <= 2000) \
                     .drop("rn")

after = all_text.count()
print(f"Avant nettoyage : {before:,} | Après : {after:,} | Supprimés : {before - after:,}")
all_text.groupBy("platform").count().orderBy(F.desc("count")).show()

Avant nettoyage : 7,063 | Après : 7,033 | Supprimés : 30


+--------+-----+
|platform|count|
+--------+-----+
|  google| 2000|
| spotify| 2000|
| youtube| 1986|
| netflix|  603|
|  chrome|  223|
|   apple|  216|
|  amazon|    5|
+--------+-----+



In [11]:
# ── A2. PIPELINE TF-IDF ──────────────────────────────────────────────────────
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, CountVectorizer, IDF, Normalizer

STOPWORDS_FR = [
    "le", "la", "les", "de", "du", "des", "un", "une", "et", "en", "a", "au",
    "pour", "par", "sur", "avec", "dans", "qui", "que", "se", "il", "elle",
    "on", "je", "tu", "nous", "vous", "ils", "elles", "est", "sont",
    "ce", "son", "sa", "ses", "mon", "ma", "mes", "ton", "ta", "tes",
    "this", "the", "of", "in", "to", "and", "is", "for", "with",
    "official", "video", "youtube", "episode", "season", "saison"
]

tokenizer  = Tokenizer(inputCol="text", outputCol="words_raw")
remover    = StopWordsRemover(
    inputCol="words_raw", outputCol="words",
    stopWords=StopWordsRemover.loadDefaultStopWords("english") + STOPWORDS_FR
)
cv         = CountVectorizer(inputCol="words", outputCol="raw_features", vocabSize=3000, minDF=5.0)
idf        = IDF(inputCol="raw_features", outputCol="tfidf_features", minDocFreq=5)
normalizer = Normalizer(inputCol="tfidf_features", outputCol="features", p=2.0)

tfidf_pipeline = Pipeline(stages=[tokenizer, remover, cv, idf, normalizer])

print("Fitting TF-IDF pipeline...")
tfidf_model = tfidf_pipeline.fit(all_text)
tfidf_df    = tfidf_model.transform(all_text)
print(f"Vocabulaire final : {len(tfidf_model.stages[2].vocabulary):,} termes")
tfidf_df.select("text", "features").show(3, truncate=60)


Fitting TF-IDF pipeline...


Vocabulaire final : 609 termes


+------------+---------------+
|        text|       features|
+------------+---------------+
|Électronique|    (609,[],[])|
|      Livres|    (609,[],[])|
|       Autre|(609,[4],[1.0])|
+------------+---------------+
only showing top 3 rows



In [12]:
# ── A3. KMEANS CONTENU (k=8) ──────────────────────────────────────────────────
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

K_CONTENT = 15

kmeans_content = KMeans(
    featuresCol="features",
    predictionCol="content_cluster",
    k=K_CONTENT,
    seed=42,
    maxIter=50
)

print(f"Training K-Means content (k={K_CONTENT})...")
km_content_model = kmeans_content.fit(tfidf_df)
content_df = km_content_model.transform(tfidf_df)

# Silhouette score
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="content_cluster")
silhouette = evaluator.evaluate(content_df)
print(f"Silhouette Score (content): {silhouette:.4f}")

# Distribution par cluster
content_df.groupBy("content_cluster").count().orderBy("content_cluster").show()

Training K-Means content (k=15)...


Silhouette Score (content): 0.4428


+---------------+-----+
|content_cluster|count|
+---------------+-----+
|              0| 6561|
|              1|   40|
|              2|   93|
|              3|   17|
|              4|   14|
|              5|    9|
|              6|   30|
|              7|    9|
|              8|   20|
|              9|   10|
|             10|   55|
|             11|  134|
|             12|    7|
|             13|    6|
|             14|   28|
+---------------+-----+



In [13]:
# ── A4. TOP TERMES PAR CLUSTER CONTENU ───────────────────────────────────────
import numpy as np
from pyspark.ml.stat import Summarizer

cv_model = tfidf_model.stages[2]
vocab    = cv_model.vocabulary

content_cluster_info = []

for cluster_id in range(K_CONTENT):
    cluster_subset = content_df.filter(F.col("content_cluster") == cluster_id)
    count = cluster_subset.count()

    top_platforms = (
        cluster_subset.groupBy("platform").count()
        .orderBy(F.desc("count")).limit(3)
        .select("platform").rdd.flatMap(lambda x: x).collect()
    )

    # Summarizer.mean aggrège les vecteurs sparse TF-IDF nativement
    mean_vec    = cluster_subset.select(Summarizer.mean(F.col("tfidf_features"))).collect()[0][0]
    top_indices = np.argsort(mean_vec.toArray())[::-1][:15]
    top_terms   = [vocab[i] for i in top_indices if i < len(vocab)]

    sample_texts = (
        cluster_subset.orderBy(F.desc("weight"))
        .select("text").limit(5)
        .rdd.flatMap(lambda x: x).collect()
    )

    content_cluster_info.append({
        "cluster_id":    cluster_id,
        "item_count":    count,
        "top_terms":     top_terms,
        "top_platforms": top_platforms,
        "sample_texts":  sample_texts
    })

    print(f"[Cluster {cluster_id}] {count:,} items | Plateformes: {top_platforms}")
    print(f"  Top termes : {', '.join(top_terms[:10])}")
    print(f"  Exemples   : {sample_texts[:3]}")


[Cluster 0] 6,561 items | Plateformes: ['spotify', 'google', 'youtube']
  Top termes : |, -, &, dj, , à, ?, !, :, 2
  Exemples   : ['Beauté/Santé', 'Livres', 'Musique/DJ']


[Cluster 1] 40 items | Plateformes: ['youtube', 'chrome', 'spotify']
  Top termes : x, -, tiakola, remix, , /, lil, dj, ), feat.
  Exemples   : ['Log in to X / X', 'X. It’s what’s happening / X', 'Profile / X']


[Cluster 2] 93 items | Plateformes: ['youtube', 'chrome', 'apple']
  Top termes : -, fr, befr, 16x9, 15sec, 6sec, lego, nl, , 30s
  Exemples   : ['Metastream Remote - Chrome\xa0Web\xa0Store', 'Login - OpenAI', 'Video DownloadHelper - CoApp Installation']


[Cluster 3] 17 items | Plateformes: ['youtube', 'google', 'spotify']
  Top termes : free, data, virtual, font, 2025, |, #automobile, dean, sea, tony
  Exemples   : ['Arielle Free', 'Free Nationals', 'how to steal / copy ( dex ) any roblox game for free 2024']


[Cluster 4] 14 items | Plateformes: ['google', 'youtube', 'chrome']
  Top termes : nuit, anyme, toute, bruxelles, (2025), business, gp, pire, train, nozgap
  Exemples   : ['Nozgap - La Nuit des Clowns (2025)', 'La nuit porte conseil in english translation', 'La 8e Nuit']


...

[Cluster 10] 55 items | Plateformes: ['chrome', 'apple', 'google']
  Top termes : google, recherche, -, navigation/utilitaires, stuttgart, maps, spa, offre, nation, nozgap
  Exemples   : ['eurotrail - Recherche Google', 'liege stuttgart train - Recherche Google', 'diskmgmt.msc - Recherche Google']


[Cluster 11] 134 items | Plateformes: ['apple', 'amazon']
  Top termes : autre, &, -, microsoft, live, app, pdf, travel, jbl, ai
  Exemples   : ['Autre', 'Xceed - We Go Out. Autre', 'Finary: Budget & Money Tracker Autre']


[Cluster 12] 7 items | Plateformes: ['youtube', 'google', 'netflix']
  Top termes : 9, 16, us, ios, super, 25, 6sec, 6s, fr, face
  Exemples   : ['9 pounds en euros', 'Better Than Us', 'FR yuki geen magie 16 9 6s BUMPER']


[Cluster 13] 6 items | Plateformes: ['google', 'youtube', 'apple']
  Top termes : flight, air, book, france, autre, -, max, funk, guide, fait
  Exemples   : ['Air France - Book a flight Autre', 'air force 667', 'open air stuttgart']


[Cluster 14] 28 items | Plateformes: ['youtube', 'google', 'netflix']
  Top termes : jeu, nouveau, fou, c'est, comment, !, ?, note, veut, 😂
  Exemples   : ['Jeu de la vie Autre', 'police d%27ecriture jeu', 'Le jeu de la dame']


In [ ]:
# ── A5. LABELLING MANUEL DES CLUSTERS CONTENU ────────────────────────────────
# Labels basés sur le run k=15 (après filtres anti-pollution + Gmail + cap 2000/platform)
# NOTE : content clustering utilisé pour référence — home page s'appuiera
#        sur l'enrichissement CATEGORY_KEYWORDS + données Delta (Part B prime)

CONTENT_LABELS = {
    0:  {"label": "🌀 Divers",              "emoji": "🌀"},   # catch-all structurel
    1:  {"label": "🎵 Musique FR",          "emoji": "🎵"},
    2:  {"label": "💻 Extensions & Outils", "emoji": "💻"},
    3:  {"label": "🖥️ Tech & Gratuit",      "emoji": "🖥️"},
    4:  {"label": "🌃 Sorties Bruxelles",   "emoji": "🌃"},
    5:  {"label": "🛍️ Shopping",            "emoji": "🛍️"},
    6:  {"label": "🎌 Manga & Web",         "emoji": "🎌"},
    7:  {"label": "🍎 Apple & Apps",        "emoji": "🍎"},
    8:  {"label": "🌍 Belgique & Culture",  "emoji": "🌍"},
    9:  {"label": "🌀 Divers",              "emoji": "🌀"},
    10: {"label": "🎓 Études & Voyage",     "emoji": "🎓"},
    11: {"label": "📱 Apps & Productivité", "emoji": "📱"},
    12: {"label": "🌀 Divers",              "emoji": "🌀"},
    13: {"label": "✈️ Voyage",              "emoji": "✈️"},
    14: {"label": "🎮 Gaming",              "emoji": "🎮"},
}

print("Labels définis. Continuer vers A6.")


In [ ]:
# ── A6. ECRITURE content_clusters ─────────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType, LongType

rows = []
for info in content_cluster_info:
    cid = info["cluster_id"]
    rows.append((
        cid,
        CONTENT_LABELS.get(cid, {}).get("label", f"Cluster {cid}"),
        CONTENT_LABELS.get(cid, {}).get("emoji", "❓"),
        info["top_terms"],
        info["top_platforms"],
        info["sample_texts"],
        info["item_count"]
    ))

schema = StructType([
    StructField("cluster_id",    IntegerType(), False),
    StructField("label",         StringType(),  False),
    StructField("emoji",         StringType(),  True),
    StructField("top_keywords",  ArrayType(StringType()), True),
    StructField("top_platforms", ArrayType(StringType()), True),
    StructField("sample_items",  ArrayType(StringType()), True),
    StructField("item_count",    LongType(),    True),
])

content_clusters_df = spark.createDataFrame(rows, schema)

out_path = os.path.join(WAREHOUSE, "content_clusters")
content_clusters_df.write.mode("overwrite").parquet(out_path)

print(f"Ecrit dans : {out_path}")
content_clusters_df.show(truncate=60)

---
## PARTIE B — Behavioral Clustering

On rassemble toutes les activités avec leurs features temporelles : heure, jour de la semaine, plateforme, poids d'interaction.

In [ ]:
# ── B1. CHARGEMENT DES FEATURES COMPORTEMENTALES ──────────────────────────────
# On utilise all_text déjà chargé (contient hour, weekday, platform, weight)
# + on ajoute tiktok_watch et instagram_likes (pas de texte utile mais données temporelles)

tiktok = read_table("tiktok_watch") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("tiktok").alias("platform"),
            F.col("interaction_weight").alias("weight"),
            F.lit(None).cast(StringType()).alias("text"))

ig_likes = read_table("instagram_likes") \
    .select(F.col("event_hour").alias("hour"),
            F.col("event_weekday").alias("weekday"),
            F.lit("instagram").alias("platform"),
            F.col("interaction_weight").alias("weight"),
            F.lit(None).cast(StringType()).alias("text"))

# Union complète pour le behavioral
behavioral_raw = all_text.union(tiktok).union(ig_likes) \
    .select("hour", "weekday", "platform", "weight") \
    .filter(F.col("hour").isNotNull() & F.col("weekday").isNotNull())

print(f"Total events comportementaux : {behavioral_raw.count():,}")
behavioral_raw.groupBy("platform").count().orderBy(F.desc("count")).show()

In [ ]:
# ── B2. FEATURE ENGINEERING COMPORTEMENTAL ────────────────────────────────────
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

# Encoder la plateforme (catégorielle → numérique)
indexer = StringIndexer(inputCol="platform", outputCol="platform_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCol="platform_idx", outputCol="platform_ohe", dropLast=False)

# Normaliser l'heure (0-23 → cyclique via sin/cos pour que minuit soit proche de 23h)
behavioral_feat = behavioral_raw \
    .withColumn("hour_sin",   F.sin(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("hour_cos",   F.cos(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("weekday_sin", F.sin(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weekday_cos", F.cos(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weight_norm", F.coalesce(F.col("weight"), F.lit(1.0)).cast("double"))

# Fit indexer + encoder
indexer_model = indexer.fit(behavioral_feat)
behavioral_feat = indexer_model.transform(behavioral_feat)

encoder_model = encoder.fit(behavioral_feat)
behavioral_feat = encoder_model.transform(behavioral_feat)

# VectorAssembler : hour_sin/cos + weekday_sin/cos + weight + platform_ohe
assembler_beh = VectorAssembler(
    inputCols=["hour_sin", "hour_cos", "weekday_sin", "weekday_cos", "weight_norm", "platform_ohe"],
    outputCol="beh_raw_features"
)

behavioral_feat = assembler_beh.transform(behavioral_feat)

# Standardisation
scaler_beh = StandardScaler(inputCol="beh_raw_features", outputCol="beh_features",
                             withMean=False, withStd=True)
scaler_model = scaler_beh.fit(behavioral_feat)
behavioral_feat = scaler_model.transform(behavioral_feat)

print("Features comportementales construites.")
behavioral_feat.select("hour", "weekday", "platform", "beh_features").show(5, truncate=80)

In [ ]:
# ── B3. KMEANS COMPORTEMENTAL (k=6) ───────────────────────────────────────────
K_BEHAVIORAL = 6

kmeans_beh = KMeans(
    featuresCol="beh_features",
    predictionCol="beh_cluster",
    k=K_BEHAVIORAL,
    seed=42,
    maxIter=50
)

print(f"Training K-Means behavioral (k={K_BEHAVIORAL})...")
km_beh_model = kmeans_beh.fit(behavioral_feat)
beh_df = km_beh_model.transform(behavioral_feat)

# Silhouette
evaluator_beh = ClusteringEvaluator(featuresCol="beh_features", predictionCol="beh_cluster")
sil_beh = evaluator_beh.evaluate(beh_df)
print(f"Silhouette Score (behavioral): {sil_beh:.4f}")

beh_df.groupBy("beh_cluster").count().orderBy("beh_cluster").show()

In [ ]:
# ── B4. CARACTERISATION DES CLUSTERS COMPORTEMENTAUX ─────────────────────────
# Pour chaque cluster : heure médiane, jour modal, plateforme dominante

beh_summary = beh_df.groupBy("beh_cluster").agg(
    F.round(F.avg("hour"), 1).alias("avg_hour"),
    F.round(F.avg("weekday"), 1).alias("avg_weekday"),
    F.count("*").alias("item_count")
).orderBy("beh_cluster")

beh_summary.show()

# Top plateforme par cluster
platform_labels = indexer_model.labels  # mapping index → platform name

beh_cluster_info = []
for cluster_id in range(K_BEHAVIORAL):
    subset = beh_df.filter(F.col("beh_cluster") == cluster_id)
    count  = subset.count()
    avg_h  = subset.agg(F.avg("hour")).collect()[0][0]
    avg_wd = subset.agg(F.avg("weekday")).collect()[0][0]
    
    top_plt = (
        subset.groupBy("platform").count()
        .orderBy(F.desc("count")).limit(3)
        .select("platform").rdd.flatMap(lambda x: x).collect()
    )
    
    # Heure humaine
    h = round(avg_h) if avg_h else 0
    if 5 <= h < 12:    period = "Matin"
    elif 12 <= h < 18: period = "Après-midi"
    elif 18 <= h < 23: period = "Soir"
    else:              period = "Nuit"
    
    wd = round(avg_wd) if avg_wd else 0
    day_type = "Weekend" if wd >= 5 else "Semaine"
    
    beh_cluster_info.append({
        "cluster_id": cluster_id,
        "item_count": count,
        "avg_hour": round(avg_h, 1) if avg_h else 0.0,
        "avg_weekday": round(avg_wd, 1) if avg_wd else 0.0,
        "time_period": period,
        "day_type": day_type,
        "top_platforms": top_plt
    })
    
    print(f"[Beh Cluster {cluster_id}] {count:,} items | {period} · {day_type} | Plateformes: {top_plt}")

In [ ]:
# ── B5. LABELLING DES CLUSTERS COMPORTEMENTAUX ────────────────────────────────
# À adapter après avoir regardé l'output ci-dessus !

BEH_LABELS = {
    0: {"label": "☀️ Journée active",      "emoji": "☀️"},
    1: {"label": "🌙 Nuit créative",        "emoji": "🌙"},
    2: {"label": "🎓 Mode étude",           "emoji": "🎓"},
    3: {"label": "🛋️ Loisirs soir",         "emoji": "🛋️"},
    4: {"label": "📅 Weekend détente",      "emoji": "📅"},
    5: {"label": "🛍️ Shopping & découverte","emoji": "🛍️"},
}

print("Labels définis. Mettre à jour BEH_LABELS après analyse ci-dessus.")

In [ ]:
# ── B6. ECRITURE behavioral_clusters ──────────────────────────────────────────
from pyspark.sql.types import DoubleType

beh_rows = []
for info in beh_cluster_info:
    cid = info["cluster_id"]
    beh_rows.append((
        cid,
        BEH_LABELS.get(cid, {}).get("label", f"Profil {cid}"),
        BEH_LABELS.get(cid, {}).get("emoji", "❓"),
        float(info["avg_hour"]),
        float(info["avg_weekday"]),
        info["time_period"],
        info["day_type"],
        info["top_platforms"],
        info["item_count"]
    ))

schema_beh = StructType([
    StructField("cluster_id",    IntegerType(), False),
    StructField("label",         StringType(),  False),
    StructField("emoji",         StringType(),  True),
    StructField("avg_hour",      DoubleType(),  True),
    StructField("avg_weekday",   DoubleType(),  True),
    StructField("time_period",   StringType(),  True),
    StructField("day_type",      StringType(),  True),
    StructField("top_platforms", ArrayType(StringType()), True),
    StructField("item_count",    LongType(),    True),
])

beh_clusters_df = spark.createDataFrame(beh_rows, schema_beh)

out_path_beh = os.path.join(WAREHOUSE, "behavioral_clusters")
beh_clusters_df.write.mode("overwrite").parquet(out_path_beh)

print(f"Ecrit dans : {out_path_beh}")
beh_clusters_df.show(truncate=50)

---
## PARTIE C — Fusion → interest_profiles

On fusionne les deux types de clusters pour créer un profil enrichi par cluster de contenu : `label_contenu · profil_comportemental_dominant`.

La home page lira cette table.

In [ ]:
# ── C1. FUSION : QUEL PROFIL COMPORTEMENTAL DOMINE CHAQUE CLUSTER CONTENU ? ───
# On joint content_cluster + beh_cluster sur les mêmes events
# Pour cela, on refait l'inférence sur les données communes (all_text)

# Ajouter le cluster contenu sur all_text
content_labeled = km_content_model.transform(tfidf_df) \
    .select("text", "hour", "weekday", "platform", "weight", "content_cluster")

# Ajouter les features comportementales puis le beh_cluster sur le même DF
# On applique les mêmes transformations que partie B
fused = content_labeled \
    .withColumn("hour_sin",    F.sin(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("hour_cos",    F.cos(2 * 3.14159 * F.col("hour") / 24)) \
    .withColumn("weekday_sin", F.sin(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weekday_cos", F.cos(2 * 3.14159 * F.col("weekday") / 7)) \
    .withColumn("weight_norm", F.coalesce(F.col("weight"), F.lit(1.0)).cast("double"))

fused = indexer_model.transform(fused)
fused = encoder_model.transform(fused)
fused = assembler_beh.transform(fused)
fused = scaler_model.transform(fused)
fused = km_beh_model.transform(fused)

print("Fusion contenu + comportemental ok.")
fused.select("text", "content_cluster", "beh_cluster").show(10, truncate=50)

In [ ]:
# ── C2. CREATION DES PROFILS FUSIONNÉS ────────────────────────────────────────
# Pour chaque content_cluster → trouver le beh_cluster dominant

cross_tab = fused.groupBy("content_cluster", "beh_cluster").count()

# beh_cluster dominant par content_cluster (argmax)
from pyspark.sql.window import Window
w = Window.partitionBy("content_cluster").orderBy(F.desc("count"))
dominant_beh = cross_tab.withColumn("rank", F.rank().over(w)) \
    .filter(F.col("rank") == 1) \
    .select("content_cluster", "beh_cluster").withColumnRenamed("beh_cluster", "dominant_beh_cluster")

dominant_beh.show()

# Construire la table interest_profiles
interest_rows = []
for row in dominant_beh.collect():
    cid  = row["content_cluster"]
    bcid = row["dominant_beh_cluster"]
    
    content_info = content_cluster_info[cid]
    beh_info     = beh_cluster_info[bcid]
    
    c_label  = CONTENT_LABELS.get(cid,  {}).get("label", f"Cluster {cid}")
    b_label  = BEH_LABELS.get(bcid,    {}).get("label", f"Profil {bcid}")
    b_emoji  = BEH_LABELS.get(bcid,    {}).get("emoji", "")
    
    # Label hybride : "🎵 Musique · Nuit créative"
    hybrid_label = f"{c_label} · {beh_info['time_period']}"
    
    interest_rows.append((
        cid,
        c_label,
        b_label,
        hybrid_label,
        CONTENT_LABELS.get(cid, {}).get("emoji", "❓"),
        content_info["top_terms"][:10],
        content_info["top_platforms"],
        content_info["sample_texts"][:5],
        round(beh_info["avg_hour"], 1),
        beh_info["time_period"],
        beh_info["day_type"],
        content_info["item_count"]
    ))
    
    print(f"Profile {cid}: {hybrid_label}")
    print(f"  Keywords: {content_info['top_terms'][:8]}")

In [ ]:
# ── C3. ECRITURE interest_profiles ────────────────────────────────────────────
schema_ip = StructType([
    StructField("cluster_id",       IntegerType(), False),
    StructField("content_label",    StringType(),  False),
    StructField("behavioral_label", StringType(),  True),
    StructField("hybrid_label",     StringType(),  True),
    StructField("emoji",            StringType(),  True),
    StructField("keywords",         ArrayType(StringType()), True),
    StructField("top_platforms",    ArrayType(StringType()), True),
    StructField("sample_items",     ArrayType(StringType()), True),
    StructField("avg_hour",         DoubleType(),  True),
    StructField("time_period",      StringType(),  True),
    StructField("day_type",         StringType(),  True),
    StructField("item_count",       LongType(),    True),
])

interest_profiles_df = spark.createDataFrame(interest_rows, schema_ip)

out_path_ip = os.path.join(WAREHOUSE, "interest_profiles")
interest_profiles_df.write.mode("overwrite").parquet(out_path_ip)

print(f"Ecrit dans : {out_path_ip}")
interest_profiles_df.select("cluster_id", "hybrid_label", "keywords", "time_period", "item_count") \
    .show(truncate=60)

---
## PARTIE D — Visualisation PCA 2D (Galaxie)

Projection PCA des clusters de contenu pour visualisation.

In [ ]:
# ── D1. PCA 2D SUR LES CLUSTERS CONTENU ───────────────────────────────────────
from pyspark.ml.feature import PCA

pca = PCA(k=2, inputCol="features", outputCol="pca_features")
pca_model = pca.fit(content_df)
pca_df = pca_model.transform(content_df)

print(f"Variance expliquée par 2 composantes PCA: {sum(pca_model.explainedVariance):.2%}")

# Extraire les coordonnées x, y pour chaque event (sample de 5000 pour la visu)
pca_sample = pca_df.select(
    F.col("content_cluster"),
    F.col("platform"),
    pca_df["pca_features"][0].alias("pca_x"),
    pca_df["pca_features"][1].alias("pca_y")
).sample(False, 0.02, seed=42).limit(5000)

pca_out = os.path.join(WAREHOUSE, "pca_viz")
pca_sample.write.mode("overwrite").parquet(pca_out)

print(f"Données PCA écrites dans : {pca_out}")
pca_sample.show(5)

In [ ]:
# ── D2. VISUALISATION MATPLOTLIB (optionnel, pour le notebook) ────────────────
try:
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import numpy as np

    pdf = pca_sample.toPandas()

    colors = cm.tab10(np.linspace(0, 1, K_CONTENT))
    fig, ax = plt.subplots(figsize=(10, 8))

    for cid in range(K_CONTENT):
        mask = pdf["content_cluster"] == cid
        label = CONTENT_LABELS.get(cid, {}).get("label", f"C{cid}")
        ax.scatter(pdf[mask]["pca_x"], pdf[mask]["pca_y"],
                   c=[colors[cid]], label=label, alpha=0.4, s=8)

    ax.set_title("MyDigitalTwin — Galaxie des centres d'intérêts (PCA 2D)", fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    plt.tight_layout()
    plt.savefig(os.path.join(WAREHOUSE, "../../../scripts/notebooks", "app", "assets", "pca_galaxy.png"), dpi=120)
    plt.show()
    print("Galaxie sauvegardée dans app/assets/pca_galaxy.png")
except ImportError:
    print("matplotlib non disponible — skipping visualization")

---
## Résumé des outputs

| Table | Contenu | Utilisée par |
|---|---|---|
| `warehouse/content_clusters` | 8 clusters TF-IDF avec top_keywords | home.py (orbit tags) |
| `warehouse/behavioral_clusters` | 6 profils temporels | home.py (section profils) |
| `warehouse/interest_profiles` | Fusion : label hybride + keywords + heure | home.py (remplacement CATEGORY_KEYWORDS) |
| `warehouse/pca_viz` | Coordonnées 2D PCA (5k points) | Dashboard visualisation |

**Prochaine étape** : mettre à jour `app/pages/home.py` pour lire `interest_profiles` au lieu du dict `CATEGORY_KEYWORDS` codé en dur.

In [ ]:
spark.stop()
print("Spark session fermée. Notebook terminé.")